# CatBoost Model Training
## Merge Silver + Gold Data and Train Model

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


### 2. Load Silver (Cleaned) Data

In [2]:
# Load cleaned datasets from silver layer
transactions = pd.read_csv("../data/silver/clean_transactions.csv")
outlets = pd.read_csv("../data/silver/clean_outlet_master.csv")
locations = pd.read_csv("../data/silver/clean_outletCoordinates.csv")
seasonality = pd.read_csv("../data/silver/clean_seasonality.csv")
holidays = pd.read_csv("../data/silver/clean_holidays.csv")

print(f"Transactions: {transactions.shape}")
print(f"Outlets: {outlets.shape}")
print(f"Locations: {locations.shape}")
print(f"Seasonality: {seasonality.shape}")
print(f"Holidays: {holidays.shape}")

Transactions: (1039959, 7)
Outlets: (20000, 4)
Locations: (19760, 3)
Seasonality: (360, 5)
Holidays: (78, 3)


### 3. Merge All Datasets

In [12]:
# Start with transactions and merge sequentially
df = transactions.merge(
    outlets,
    left_on='gitOutlet_ID',
    right_on='Outlet_ID',
    how='left'
)
print(f"After outlets merge: {df.shape}")

# Merge locations
df = df.merge(locations, on='Outlet_ID', how='left')
print(f"After locations merge: {df.shape}")

# Merge seasonality
df = df.merge(seasonality, on=['Distributor_ID', 'Month'], how='left')
print(f"After seasonality merge: {df.shape}")

After outlets merge: (1039959, 11)
After locations merge: (1039959, 13)
After seasonality merge: (3119877, 16)


### 4. Merge with Gold Features

In [11]:
# Load and merge gold features
gold_features = pd.read_csv("../data/gold/features.csv")
print(f"Gold features shape: {gold_features.shape}")
print(f"Gold features columns: {gold_features.columns.tolist()}")

df = df.merge(gold_features, on='gitOutlet_ID', how='left')
print(f"After gold features merge: {df.shape}")

Gold features shape: (20000, 6)
Gold features columns: ['gitOutlet_ID', 'volume_mean', 'volume_sum', 'transactions', 'Potential_Target', 'target']
After gold features merge: (3119877, 23)


### 5. Data Cleanup

In [13]:
# Debug: Check all columns before cleanup
print("All columns before cleanup:")
print(df.columns.tolist())
print(f"\nDataframe shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

All columns before cleanup:
['gitOutlet_ID', 'Year_x', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value', 'Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type', 'Latitude', 'Longitude', 'Unnamed: 0', 'Year_y', 'Seasonality_Index']

Dataframe shape: (3119877, 16)

First few rows:
  gitOutlet_ID  Year_x  Month Distributor_ID  SKU_ID  Volume_Liters  \
0    OUT_19886    2024     12      DIST_S_02  SKU_10       5.897879   
1    OUT_19886    2024     12      DIST_S_02  SKU_10       5.897879   
2    OUT_19886    2024     12      DIST_S_02  SKU_10       5.897879   
3    OUT_00837    2024      2      DIST_W_01  SKU_03      20.697364   
4    OUT_00837    2024      2      DIST_W_01  SKU_03      20.697364   

   Total_Bill_Value  Outlet_ID Outlet_Size  Cooler_Count Outlet_Type  \
0       2177.632359  OUT_19886       Small             0       Hotel   
1       2177.632359  OUT_19886       Small             0       Hotel   
2       2177.632359  OUT_19886       Small        

In [14]:
# Handle duplicate columns from merges
if 'Latitude_y' in df.columns:
    df = df.drop(columns=['Latitude_y', 'Longitude_y'], errors='ignore')
    if 'Latitude_x' in df.columns:
        df = df.rename(columns={'Latitude_x': 'Latitude', 'Longitude_x': 'Longitude'})

if 'Outlet_ID_y' in df.columns:
    df = df.drop(columns=['Outlet_ID_y'])
if 'Outlet_ID_x' in df.columns:
    df = df.rename(columns={'Outlet_ID_x': 'Outlet_ID'})

# Rename Year_x to Year and drop Year_y
if 'Year_x' in df.columns:
    df = df.rename(columns={'Year_x': 'Year'})
if 'Year_y' in df.columns:
    df = df.drop(columns=['Year_y'])

# Drop unnecessary columns
cols_to_drop = ['Unnamed: 0', 'Seasonality_Index']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# Create date column
df['date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' +
    df['Month'].astype(str) + '-01',
    errors='coerce'
)

print(f"Data shape after cleanup: {df.shape}")
print(f"Columns: {df.columns.tolist()[:20]}...")  # Show first 20

Data shape after cleanup: (3119877, 14)
Columns: ['gitOutlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value', 'Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type', 'Latitude', 'Longitude', 'date']...


### 6. Feature Engineering

In [15]:
# Sort by outlet and date for time series features
df = df.sort_values(['gitOutlet_ID', 'date']).reset_index(drop=True)

# Time-based features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['quarter'] = df['date'].dt.quarter
df['dayofweek'] = df['date'].dt.dayofweek
df['weekofyear'] = df['date'].dt.isocalendar().week
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# Cyclical encoding for month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Lag features
df['lag_1'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(1)
df['lag_3'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(3)
df['lag_6'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(6)
df['lag_12'] = df.groupby('gitOutlet_ID')['Volume_Liters'].shift(12)

# Rolling mean features
df['rolling_mean_3'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['rolling_mean_6'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform(lambda x: x.rolling(6, min_periods=1).mean())
df['rolling_mean_12'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform(lambda x: x.rolling(12, min_periods=1).mean())

# Aggregated features
df['outlet_avg_volume'] = df.groupby('gitOutlet_ID')['Volume_Liters'].transform('mean')
df['product_avg_volume'] = df.groupby('SKU_ID')['Volume_Liters'].transform('mean')
df['distributor_avg_volume'] = df.groupby('Distributor_ID')['Volume_Liters'].transform('mean')

# Holiday feature
holidays['Date'] = pd.to_datetime(holidays['Date'])
holiday_months = holidays['Date'].dt.month.unique()
df['is_holiday_month'] = df['month'].isin(holiday_months).astype(int)

# Geographic clustering
coords = df[['Latitude', 'Longitude']].dropna()
if len(coords) > 0:
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
    df.loc[coords.index, 'geo_cluster'] = kmeans.fit_predict(coords)
    df['geo_cluster'] = df['geo_cluster'].fillna(-1).astype(int)

print("Feature engineering completed")
print(f"Total features created: {df.shape[1]}")

Feature engineering completed
Total features created: 35


### 7. Prepare Features and Target

In [16]:
# Drop rows with missing target or lag features
df_clean = df.dropna(subset=['Volume_Liters', 'lag_1']).copy()
print(f"Data shape after dropping NaNs: {df_clean.shape}")

# Select features (numeric only)
exclude_cols = ['Volume_Liters', 'Total_Bill_Value', 'date', 'Latitude', 'Longitude', 'Year']
feature_cols = [col for col in df_clean.columns if col not in exclude_cols and df_clean[col].dtype in ['int64', 'float64']]
feature_cols = [col for col in feature_cols if col not in df_clean.select_dtypes(include='object').columns]

X = df_clean[feature_cols].fillna(0)
y = df_clean['Volume_Liters']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns (first 15): {feature_cols[:15]}")

Data shape after dropping NaNs: (3099877, 35)
Features shape: (3099877, 17)
Target shape: (3099877,)
Number of features: 17
Feature columns (first 15): ['Month', 'Cooler_Count', 'is_weekend', 'month_sin', 'month_cos', 'lag_1', 'lag_3', 'lag_6', 'lag_12', 'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12', 'outlet_avg_volume', 'product_avg_volume', 'distributor_avg_volume']


### 8. Train-Test Split

In [17]:
# Use temporal split: train on earlier data, test on 2025 data
train_mask = df_clean['year'] < 2025
test_mask = df_clean['year'] == 2025

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train samples: {len(y_train)}, Test samples: {len(y_test)}")

Train shape: (2057572, 17)
Test shape: (1042305, 17)
Train samples: 2057572, Test samples: 1042305


### 9. Install and Import CatBoost

In [18]:
try:
    from catboost import CatBoostRegressor
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost"])
    from catboost import CatBoostRegressor

print("CatBoost imported successfully")

CatBoost imported successfully


### 10. Train CatBoost Model

In [19]:
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_state=42,
    verbose=50,
    task_type='CPU',
    thread_count=-1
)

print("Training CatBoost model...")
model.fit(
    X_train,
    y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)
print("Model training completed!")

Training CatBoost model...
0:	learn: 91.1914562	test: 90.8892346	best: 90.8892346 (0)	total: 551ms	remaining: 4m 34s
50:	learn: 18.1801965	test: 18.2115115	best: 18.2115115 (50)	total: 11.8s	remaining: 1m 44s
100:	learn: 12.4245367	test: 12.4795319	best: 12.4795319 (100)	total: 32.4s	remaining: 2m 8s
150:	learn: 10.5613782	test: 10.6303039	best: 10.6303039 (150)	total: 56.7s	remaining: 2m 11s
200:	learn: 9.5601945	test: 9.6244690	best: 9.6244690 (200)	total: 1m 8s	remaining: 1m 42s
250:	learn: 8.8027806	test: 8.8757502	best: 8.8757502 (250)	total: 1m 27s	remaining: 1m 26s
300:	learn: 8.2410063	test: 8.3228007	best: 8.3228007 (300)	total: 1m 40s	remaining: 1m 6s
350:	learn: 7.8613680	test: 7.9486675	best: 7.9486675 (350)	total: 1m 54s	remaining: 48.5s
400:	learn: 7.5478635	test: 7.6398131	best: 7.6398131 (400)	total: 2m 13s	remaining: 32.9s
450:	learn: 7.2942164	test: 7.3920856	best: 7.3920856 (450)	total: 2m 29s	remaining: 16.2s
499:	learn: 7.0861207	test: 7.1880853	best: 7.1880853 (49

### 11. Model Evaluation

In [20]:
# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("="*50)
print("MODEL EVALUATION METRICS")
print("="*50)
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")
print(f"Train MAE:  {train_mae:.4f}")
print(f"Test MAE:   {test_mae:.4f}")
print(f"Train R²:   {train_r2:.4f}")
print(f"Test R²:    {test_r2:.4f}")
print("="*50)

MODEL EVALUATION METRICS
Train RMSE: 7.0861
Test RMSE:  7.1881
Train MAE:  3.3814
Test MAE:   3.4006
Train R²:   0.9945
Test R²:    0.9943


### 12. Feature Importance

In [21]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print("="*50)
print(feature_importance.head(20).to_string(index=False))
print("="*50)


Top 20 Most Important Features:
               feature  importance
    product_avg_volume   53.961285
     outlet_avg_volume   26.765454
        rolling_mean_3    8.065255
                 lag_1    5.493392
             month_cos    2.326921
                 lag_3    1.051971
                 Month    0.801130
             month_sin    0.751999
distributor_avg_volume    0.338561
        rolling_mean_6    0.272427
           geo_cluster    0.095845
          Cooler_Count    0.045262
                 lag_6    0.018725
       rolling_mean_12    0.010415
                lag_12    0.001177
            is_weekend    0.000181
      is_holiday_month    0.000000


### 13. Save Predictions

In [22]:
# Get predictions for all data
y_pred_all = model.predict(X)

# Create results dataframe
results = pd.DataFrame({
    'gitOutlet_ID': df_clean['gitOutlet_ID'].values,
    'Outlet_ID': df_clean['Outlet_ID'].values,
    'date': df_clean['date'].values,
    'Actual_Volume': y.values,
    'Predicted_Volume': y_pred_all,
    'Error': (y.values - y_pred_all),
    'Absolute_Error': np.abs(y.values - y_pred_all)
})

results = results.sort_values(['gitOutlet_ID', 'date'])

# Save to CSV
results.to_csv('../outputs/catboost_predictions.csv', index=False)
print(f"Predictions saved to ../outputs/catboost_predictions.csv")
print(f"Total predictions: {len(results)}")
print(f"\nFirst 10 predictions:")
print(results.head(10).to_string())

Predictions saved to ../outputs/catboost_predictions.csv
Total predictions: 3099877

First 10 predictions:
  gitOutlet_ID  Outlet_ID       date  Actual_Volume  Predicted_Volume     Error  Absolute_Error
0    OUT_00001  OUT_00001 2023-03-01      79.993079         89.656418 -9.663339        9.663339
1    OUT_00001  OUT_00001 2023-03-01      79.993079         89.656418 -9.663339        9.663339
2    OUT_00001  OUT_00001 2023-03-01      16.425555         15.659647  0.765907        0.765907
3    OUT_00001  OUT_00001 2023-03-01      16.425555         16.224144  0.201411        0.201411
4    OUT_00001  OUT_00001 2023-03-01      16.425555         16.176302  0.249252        0.249252
5    OUT_00001  OUT_00001 2023-03-01     282.573656        286.809061 -4.235405        4.235405
6    OUT_00001  OUT_00001 2023-03-01     282.573656        285.532141 -2.958485        2.958485
7    OUT_00001  OUT_00001 2023-03-01     282.573656        287.481000 -4.907344        4.907344
8    OUT_00001  OUT_00001 202

### 14. Summary Statistics

In [23]:
print("\n" + "="*60)
print("PREDICTION SUMMARY")
print("="*60)
print(f"Mean Absolute Error: {results['Absolute_Error'].mean():.4f}")
print(f"Median Absolute Error: {results['Absolute_Error'].median():.4f}")
print(f"Max Error: {results['Absolute_Error'].max():.4f}")
print(f"Min Error: {results['Absolute_Error'].min():.4f}")
print(f"Std Dev Error: {results['Absolute_Error'].std():.4f}")
print("="*60)
print(f"Actual Volume - Mean: {results['Actual_Volume'].mean():.2f}, Std: {results['Actual_Volume'].std():.2f}")
print(f"Predicted Volume - Mean: {results['Predicted_Volume'].mean():.2f}, Std: {results['Predicted_Volume'].std():.2f}")
print("="*60)


PREDICTION SUMMARY
Mean Absolute Error: 3.3879
Median Absolute Error: 1.6378
Max Error: 352.0992
Min Error: 0.0000
Std Dev Error: 6.2630
Actual Volume - Mean: 53.17, Std: 95.52
Predicted Volume - Mean: 53.18, Std: 95.11


In [25]:
### 15. Create Latent Potential Output (January 2026)

# Check what date range we have
print("Date range in predictions:")
print(f"Min date: {results['date'].min()}")
print(f"Max date: {results['date'].max()}")
print(f"Unique years: {sorted(results['date'].dt.year.unique())}")
print(f"Unique months: {sorted(results['date'].dt.month.unique())}")

# Filter for January 2026
jan_2026 = results[(results['date'].dt.year == 2026) & (results['date'].dt.month == 1)].copy()

print(f"\nRecords for January 2026: {len(jan_2026)}")

# If no 2026 data, get the latest month available
if len(jan_2026) == 0:
    print("\nNo 2026 data found. Using latest month in dataset...")
    latest_date = results['date'].max()
    print(f"Latest date available: {latest_date}")
    jan_2026 = results[(results['date'].dt.year == latest_date.year) & 
                       (results['date'].dt.month == latest_date.month)].copy()
    print(f"Using {latest_date.strftime('%B %Y')} instead ({len(jan_2026)} records)")

# Get maximum potential per outlet
latent_potential = jan_2026.groupby('Outlet_ID').agg({
    'Predicted_Volume': 'sum'  # Sum of predicted volumes for all SKUs in the month
}).reset_index()

latent_potential.columns = ['Outlet_ID', 'Maximum_Monthly_Liters']
latent_potential = latent_potential.sort_values('Maximum_Monthly_Liters', ascending=False)

print(f"\nLatent Potential Summary:")
print(f"Number of outlets with predictions: {len(latent_potential)}")
print(f"Mean potential: {latent_potential['Maximum_Monthly_Liters'].mean():.2f}")
print(f"Max potential: {latent_potential['Maximum_Monthly_Liters'].max():.2f}")
print(f"Min potential: {latent_potential['Maximum_Monthly_Liters'].min():.2f}")

# Save to outputs
output_file = '../outputs/Data_Dynamos_predictions.csv'
latent_potential.to_csv(output_file, index=False)
print(f"\n✓ Latent Potential output saved to: {output_file}")
print(f"\nTop 10 outlets by potential:")
print(latent_potential.head(10).to_string(index=False))

Date range in predictions:
Min date: 2023-01-01 00:00:00
Max date: 2025-12-01 00:00:00
Unique years: [np.int32(2023), np.int32(2024), np.int32(2025)]
Unique months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]

Records for January 2026: 0

No 2026 data found. Using latest month in dataset...
Latest date available: 2025-12-01 00:00:00
Using December 2025 instead (87174 records)

Latent Potential Summary:
Number of outlets with predictions: 11121
Mean potential: 572.04
Max potential: 5741.22
Min potential: 10.23

✓ Latent Potential output saved to: ../outputs/Data_Dynamos_predictions.csv

Top 10 outlets by potential:
Outlet_ID  Maximum_Monthly_Liters
OUT_11029             5741.222995
OUT_03493             5549.552048
OUT_07215             5470.635661
OUT_13828             5302.773040
OUT_01207             5171.324367
OUT_07796             5161.233610
OUT_12588             51